# Reinforcement Learning with NeuralRNN

Supervised notebooks in this series train RNNs with a teacher signal at every
time step. Animals do not learn that way: they act, and much later receive a
reward or nothing at all. **Reinforcement learning (RL)** is the framework
for learning from such sparse, delayed evaluative feedback — and it is the
right tool when we want RNN models that *learn* the way animals do.

This tutorial shows how to do RL modeling of cognitive tasks with NeuralRNN,
reproducing the paradigm of **Song, Yang & Wang (2017), "Reward-based
training of recurrent neural networks for cognitive and value-based tasks"
(eLife)**.

**What you will learn**

1. how the RL layer maps onto the pieces you already know (env ↔ dataset,
   agent ↔ model, RL loss ↔ objective, `RLTrainer` ↔ `Trainer`);
2. how to get an environment: neurogym passthrough, or writing and
   registering your own;
3. how to build an agent: any RNN core + actor/critic heads
   (`ActorCriticModel`);
4. how REINFORCE with a learned value baseline works, and how the loss
   registry lets you plug in new algorithms;
5. how to analyze a reward-trained RNN: learning curves, psychometrics,
   choice-selective units, and the paper's signature *critic-lesion* control.

**Framework mapping** (supervised → RL):

| layer | supervised | reinforcement learning |
|---|---|---|
| data | `Dataset` / `load_dataset` | **env** with a gym-style `reset`/`step` contract (`make_env`, `register_env`) |
| model | `NeuralDynamicsModel` | **`ActorCriticModel`** = any RNN core + actor (policy) and critic (value) heads |
| loss | `Objective` (per-step teacher) | **RL loss** on sampled actions (`PPOLoss`, `ReinforceLoss`, `RL_LOSS_REGISTRY`) |
| trainer | `Trainer` | **`RLTrainer`** (rollout → returns → BPTT update) |


In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from neuralrnn import (
    AutoModel, ActorCriticConfig, ActorCriticModel,
    RLTrainer, RLTrainingArguments, SyncVectorEnv, make_env, register_env,
    collect_episodes, build_rl_loss,
)
from neuralrnn.models.gated_rnn import GatedRNNConfig
from neuralrnn.analysis import fit_pca

MODEL_DIR = "models/reinforcement_learning"
FIG_DIR = "figs/reinforcement_learning"
CKPT_DIR = os.path.join(MODEL_DIR, "final")
for d in (MODEL_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "font.size": 12})


def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight", dpi=150)
    print("saved", os.path.join(FIG_DIR, name))

## 1. The environment: a cognitive task as a POMDP

In RL the world is a **partially observed Markov decision process**: at each step the agent sees an observation $o_t$, picks an action $a_t \sim \pi(\cdot|h_t)$ (where $h_t$ is the agent's internal state), and receives a reward $r_t$. The agent's goal is to maximize expected return $\mathbb{E}[\sum_t r_t]$.

We use **perceptual decision making** (random-dot motion discrimination) for example: the subject fixates, watches noisy motion evidence, then reports the dominant direction.

* **Observation** — 3 channels: `fixation` (1 while the agent should hold still), `stim_L` and `stim_R`, two noisy evidence streams whose means are
  $(1 \pm c)/2$ for coherence $c$.
* **Actions** — 0 = *fixate*, 1 = *choose left*, 2 = *choose right*. Choosing during fixation aborts the trial (reward $-0.1$).
* **Reward** — $+1$ for a correct choice, $0$ for a wrong one. That is all the network ever sees: no per-step target, no teacher.

NeuralRNN envs follow the gymnasium contract (`reset() → (obs, info)`, `step(a) → (obs, reward, terminated, truncated, info)`), so **any neurogym task works out of the box** through
`make_env("neurogym:<TaskId>")`. The wrapper (`NeurogymEnvAdapter`) turns each neurogym trial into an episode. It reports `terminated=True` at the trial boundary and packs per-trial statistics into `info["episode"]`, so the trainer can reset the RNN's hidden state between trials and log returns.

We use a shortened stimulus (500 ms) and `dt = 100` ms with 7 steps per trial to keep the tutorial fast.

In [ ]:
TIMING = {"fixation": 100, "stimulus": 500, "decision": 100}  # ms

def make_pdm(seed):
    """One PDM env thunk (a callable returning a fresh env)."""
    return make_env("neurogym:PerceptualDecisionMaking-v0", dt=100,
                    timing=TIMING, seed=seed)

env = make_pdm(seed=0)()
print("observation space:", env.observation_space)
print("action space:     ", env.action_space)

obs, info = env.reset()
print("first obs (fixation, stim_L, stim_R):", obs)

In [ ]:
# Roll a few trials by hand: fixate until the decision epoch, then choose.
fig, axes = plt.subplots(3, 1, figsize=(5, 5))
for ax, seed in zip(axes, (0, 1, 2)):
    env = make_pdm(seed=seed)()
    obs, _ = env.reset()
    O, R = [obs], []
    done, t = False, 0
    evidence = np.zeros(2)      # integrated (stim_L, stim_R)
    while not done and t < 50:
        # trial = 1 fixation + 5 stimulus + 1 decision step; neurogym counts
        # the reset observation as step 0, so the decision epoch is t = 5
        if t < 5:
            a = 0                                       # keep fixating
            if t >= 1:
                evidence += obs[1:3]                    # integrate the evidence
        else:
            a = 1 if evidence[0] > evidence[1] else 2   # choose stronger side
        obs, r, term, trunc, info = env.step(a)
        O.append(obs); R.append(r)
        done = term or trunc
        t += 1
    O = np.array(O)
    tt = np.arange(len(O)) * 0.1
    ax.plot(tt, O[:, 0], "k-", label="fixation")
    ax.plot(tt, O[:, 1], "C0-", alpha=0.8, label="stim L")
    ax.plot(tt, O[:, 2], "C1-", alpha=0.8, label="stim R")
    ax2 = ax.twinx()
    ax2.step(tt[1:], R, "r-", where="post", label="reward")
    ax2.set_ylim(-0.15, 1.2)
    ax2.set_yticks([0, 1])
    ax.set_ylabel("input")
    ep = info["episode"]
    ax.set_title(f"choice={'L' if a == 1 else 'R'}  ->  "
                 f"return={ep['r']:.1f}, {'correct' if ep['performance'] else 'wrong/aborted'}",
                 fontsize=9)
axes[0].legend(fontsize=10, loc="upper right")
axes[-1].set_xlabel("time in trial (s)")
fig.suptitle("Example trials of PerceptualDecisionMaking")
fig.tight_layout()
savefig(fig, "env_trials.png")
plt.show()

### Writing and registering your own env

Any object with `observation_space`, `action_space`, `reset()` and `step()`
can be registered with `register_env(name, constructor)` and then built with
`make_env(name)` — the same pattern neurogym itself uses. Here is a complete
two-armed bandit in a few lines (one-step episodes: obs is constant, arm 1
pays 0.8, arm 2 pays 0.2):

In [ ]:
from gymnasium import spaces

class TwoArmedBandit:
    """One-step bandit: the agent sees a constant obs and pulls an arm."""
    observation_space = spaces.Box(0.0, 1.0, shape=(1,))
    action_space = spaces.Discrete(2)

    def __init__(self, p=(0.8, 0.2), seed=None):
        self.p = p
        self.rng = np.random.default_rng(seed)

    def reset(self, *, seed=None, options=None):
        return np.array([1.0], dtype=np.float32), {}

    def step(self, action):
        r = float(self.rng.random() < self.p[action])
        # one-step episode; stats for the trainer's logs
        return (np.array([1.0], dtype=np.float32), r, True, False,
                {"episode": {"r": r, "l": 1}})

register_env("bandit", lambda **kw: TwoArmedBandit(**kw))

env = make_env("bandit", seed=0)()
for _ in range(3):
    obs, _ = env.reset()
    print(env.step(1)[:3])   # always pull arm 1

The envs used by the reproduction notebooks (e.g., `18_rl_rnn_paradigmA`,
) are registered exactly this way — see
`neuralrnn/rl/envs/echoice.py` for
full-size examples.

## 2. The agent: an RNN core with actor and critic heads

The agent keeps a hidden state $h_t$ that integrates the observations, and
two linear readouts:

$$
h_t = F_\theta(h_{t-1}, o_t), \qquad
\pi_\theta(a|h_t) = \mathrm{softmax}(W_a h_t), \qquad
V_\theta(h_t) = W_v h_t .
$$

* $F_\theta$ is **any registered NeuralRNN core** — here a GRU
  (`GatedRNNConfig`), close to the leaky rectified GRU of the paper.
* $\pi$ (the **actor**) is the policy: a softmax over the 3 actions.
* $V$ (the **critic**) estimates the expected future reward; it is only
  used to *train* the actor (§3), not to act.

`ActorCriticModel` wraps this up and still satisfies the standard
`NeuralDynamicsModel` contract (`recurrence` / `readout` / `init_state`), so
every analysis tool in `neuralrnn.analysis` works on it unchanged.

*Difference from the paper:* Song et al. use two *separate* networks (a
policy GRU and a value GRU that reads the policy network's activity). A
shared core with two heads is the standard modern simplification — the
critic still reads the policy's hidden state, which is the essential point.

In [ ]:
agent = ActorCriticModel(ActorCriticConfig(
    core_config=GatedRNNConfig(input_dim=3, latent_dim=32).to_dict(),
    action_dim=3,
    action_type="discrete",
))
print(agent)
print("trainable parameters:",
      sum(p.numel() for p in agent.parameters()))

# The step interface: observation + previous hidden state -> action + new state
obs, _ = make_pdm(seed=0)().reset()
z = agent.init_state(batch_size=1, device="cpu")
x = torch.as_tensor(obs, dtype=torch.float32).view(1, -1)
action, log_prob, entropy, value, z = agent.step(x, z)
print(f"action={int(action)}  logp={float(log_prob):.2f}  "
      f"V={float(value):.2f}  z={tuple(z.shape)}")

## 3. The algorithm: REINFORCE with a learned value baseline

How do you train a policy when nobody tells you the right action? **Policy
gradient**: make rewarding trajectories more likely. The REINFORCE estimator
is

$$
\nabla_\theta J \;\approx\; \sum_t \nabla_\theta \log
\pi_\theta(a_t|h_t)\;\big(G_t - V(h_t)\big),
$$

where $G_t = \sum_{t' \ge t} r_{t'}$ is the return-to-go. The term in
parentheses is the **advantage**: how much better the outcome was than the
critic expected. Multiplying the log-prob of each action by its advantage
pushes up the probability of actions that led to unexpectedly good outcomes.

Two things matter for it to work in practice:

* **The baseline.** Raw returns are noisy; subtracting the critic's
  prediction $V(h_t)$ centers them and massively reduces the gradient
  variance. The critic is trained alongside by regressing onto the observed
  returns: $\min \sum_t (V(h_t) - G_t)^2$.
* **Backprop through time.** Gradients flow through the whole rollout, so
  the RNN can learn to *remember* evidence across the trial even though the
  reward only arrives at the end.

This is exactly the recipe of Song et al. (2017) — the simplest
actor–critic method, and the historically first one applied to cognitive
RNNs. In NeuralRNN it lives in the RL loss registry next to PPO:

| loss | registry name | use when |
|---|---|---|
| `ReinforceLoss` | `"reinforce"` | on-policy, single update per batch (`update_epochs=1, num_minibatches=1`) |
| `PPOLoss` | `"ppo"` | sample-efficient, multiple epochs per batch (default) |

New algorithms register the same way (`@register_rl_loss("my_algo")`) and
plug into `RLTrainer` unchanged — see `neuralrnn/rl/README.md`.

In [ ]:
loss = build_rl_loss("reinforce", ent_coef=0.001)
print(type(loss).__name__, "registered in RL_LOSS_REGISTRY")

# REINFORCE is strictly on-policy: one gradient step per collected batch.
args = RLTrainingArguments(
    total_timesteps=500_000,   # ~140k trials at 7 steps/trial (paper: effectively unbounded, early stop)
    num_envs=16,                 # parallel envs per batch
    num_steps=64,                # rollout length per env (covers ~9 trials)
    num_minibatches=1,           # whole batch in one piece ...
    update_epochs=1,             # ... and only one pass over it (on-policy!)
    estimator="nstep",           # returns-to-go with bootstrapping at chunk ends
    gamma=1.0,                   # no discounting inside a 7-step trial
    lr=4e-3,                     # paper: Adam lr=0.004
    device="cpu",                # tiny model: CPU is faster (kernel-launch overhead)
    eval_every=10,
    early_stop_metric="p_correct",
    output_dir=MODEL_DIR,
)

## 4. Training

`RLTrainer` runs the loop: collect a batch of rollouts with
`SyncVectorEnv`, compute returns/advantages, replay the sequences through
the RNN for the BPTT update. The agent's hidden state is reset automatically
at every trial boundary.

We evaluate every 10 updates on 200 held-out trials (`p_correct` = fraction
correct among decided trials). Training takes ~3 minutes on CPU; on reruns
the checkpoint is loaded instead.

In [ ]:
@torch.no_grad()
def eval_fn(agent, n_trials=200):
    """Held-out performance: fraction correct on fresh trials."""
    env = make_pdm(seed=10_000)()
    n_corr = 0
    for _ in range(n_trials):
        obs, _ = env.reset()
        z = agent.init_state(1, "cpu")
        done = False
        while not done:
            x = torch.as_tensor(obs, dtype=torch.float32).view(1, -1)
            a, *_, z = agent.step(x, z)
            obs, r, term, trunc, info = env.step(int(a))
            done = term or trunc
        n_corr += info["episode"]["performance"]
    return {"p_correct": n_corr / n_trials}


if os.path.exists(os.path.join(CKPT_DIR, "config.json")):
    agent = AutoModel.from_pretrained(CKPT_DIR)
    print("loaded checkpoint from", CKPT_DIR)
else:
    envs = SyncVectorEnv([make_pdm(seed=i) for i in range(args.num_envs)])
    trainer = RLTrainer(agent, envs, args, loss=loss, eval_fn=eval_fn)
    history = trainer.train()   # checkpoints: models/reinforcement_learning/{best,final}

In [ ]:
# Learning curves (history.json is written by RLTrainer during training)
%matplotlib inline
with open(os.path.join(MODEL_DIR, "history.json")) as f:
    history = json.load(f)
scalars, ev = history["scalars"], history["eval"]

fig, axes = plt.subplots(1, 3, figsize=(7, 3))
x = [s["global_step"] / 1e3 for s in scalars]
axes[0].plot(x, [s["episodic_return"] for s in scalars], lw=1)
axes[0].set_ylabel("return per trial")
axes[0].set_title("reward grows as \nthe policy improves")
axes[1].plot(x, [s["v_loss"] for s in scalars], lw=1, color="C3")
axes[1].set_ylabel("critic loss")
axes[1].set_title("the critic learns\nto predict reward")
axes[2].plot([e["update"] for e in ev], [e["p_correct"] for e in ev], "o-",
             ms=3, color="C2")
axes[2].axhline(0.5, color="k", ls="--", lw=0.8)
axes[2].set_ylabel("p(correct)")
axes[2].set_xlabel("update")
axes[2].set_title("held-out accuracy")
for ax in axes[:2]:
    ax.set_xlabel("env steps (k)")
fig.suptitle("Learning from reward only (REINFORCE on PDM)")
fig.tight_layout()
savefig(fig, "learning_curves.png")
plt.show()

## 5. Behavior: the psychometric curve

The first thing to check in a decision-making model is the **psychometric
function**: accuracy as a function of stimulus strength (coherence). A good
learner is near-perfect at high coherence and at chance when the evidence
vanishes. We run the trained agent on fresh trials and read the coherence
off the mean stimulus inputs.

In [ ]:
envs = SyncVectorEnv([make_pdm(seed=20_000 + i) for i in range(8)])
episodes = collect_episodes(agent, envs, 400, deterministic=True)
print(f"collected {len(episodes)} episodes")

cohs, correct = [], []
for ep in episodes:
    m = ep["obs"][:, 1:3].mean(axis=0)     # mean stim_L, stim_R over the trial
    if m.sum() <= 0:
        continue
    cohs.append((m[0] - m[1]) / m.sum() * 100)   # signed coherence (%)
    correct.append(ep["info"]["performance"])
cohs, correct = np.array(cohs), np.array(correct)

bins = [-100, -20, -8, -3, -0.5, 0.5, 3, 8, 20, 100]
idx = np.digitize(cohs, bins)
xs, ys, ns = [], [], []
for b in range(1, len(bins)):
    m = idx == b
    if m.sum() >= 5:
        xs.append(cohs[m].mean()); ys.append(correct[m].mean()); ns.append(m.sum())

fig, ax = plt.subplots(figsize=(3, 2))
ax.plot(xs, ys, "o-", color="C0")
ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("coherence (%, + = leftward evidence)")
ax.set_ylabel("p(correct)")
ax.set_title("Psychometric curve of the trained agent")
fig.tight_layout()
savefig(fig, "psychometric.png")
plt.show()

## 6. Neural dynamics: what did the network learn to represent?

Reward-only training should still produce *neurally plausible* activity —
that was the paper's central claim. Two quick looks:

1. **State-space geometry.** Project hidden states onto their first PCs and
   color by the final choice: the network should separate the two choices
   into distinct trajectory bundles.
2. **Choice-selective units.** Single units whose stimulus-epoch activity
   differs between left and right choices — the RL analog of the
   decision-related neurons recorded in LIP/PFC.

In [ ]:
choice = np.array([ep["actions"][-1] for ep in episodes])
Z_all = np.concatenate([ep["states"] for ep in episodes], axis=0)
pca = fit_pca(Z_all, n_components=3)
print("explained variance (3 PCs):",
      np.round(pca.explained_variance_ratio, 3))

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
ax = axes[0]
for c, color, lab in ((1, "C0", "choice L"), (2, "C1", "choice R")):
    for ep in [e for e, ch in zip(episodes, choice) if ch == c][:40]:
        Z = pca.transform(ep["states"])
        ax.plot(Z[:, 0], Z[:, 1], color=color, alpha=0.25, lw=0.8)
ax.plot([], [], color="C0", label="choice L")
ax.plot([], [], color="C1", label="choice R")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("trial trajectories by choice")
ax.legend(fontsize=8)

# choice selectivity of single units (stimulus window, steps 1..5)
ax = axes[1]
Zstim = np.stack([ep["states"][1:6].mean(axis=0) for ep in episodes])
sel = np.abs(Zstim[choice == 1].mean(0) - Zstim[choice == 2].mean(0)) / \
      (Zstim.std(0) + 1e-9)
top = np.argsort(sel)[::-1][:3]
T = max(len(ep["states"]) for ep in episodes)
tt = np.arange(T) * 0.1
for u, color in zip(top, ("C2", "C3", "C4")):
    for c, ls, lab in ((1, "-", "L"), (2, "--", "R")):
        tr = np.stack([ep["states"][:, u]
                       for ep, ch in zip(episodes, choice)
                       if ch == c and len(ep["states"]) == T])
        ax.plot(tt, tr.mean(0), ls, color=color,
                label=f"unit {u}, choice {lab}")
ax.axvspan(0.1, 0.6, color="0.9", zorder=0)
ax.set_xlabel("time in trial (s)"); ax.set_ylabel("activity")
ax.set_title("most choice-selective units")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
savefig(fig, "neural_dynamics.png")
plt.show()

## 7. The critic-lesion control

The paper's most elegant control: after training, **remove the value
network entirely** — behavior is essentially unchanged. The critic is
necessary for *learning* (it supplies the baseline that shapes the
gradient), but not for *performance* (acting only needs the policy). The
dopamine analogy is deliberate: a teaching signal, not a decision variable.

With `ActorCriticModel` the lesion is a few lines: zero the critic head and
re-measure accuracy.

In [ ]:
with torch.no_grad():
    acc_intact = eval_fn(agent)
    w, b = agent.critic.weight.clone(), agent.critic.bias.clone()
    agent.critic.weight.zero_()          # lesion: critic outputs 0 everywhere
    agent.critic.bias.zero_()
    print("value after lesion:",
          float(agent.value(agent.init_state(1, "cpu")).squeeze()))
    acc_lesioned = eval_fn(agent)
    agent.critic.weight.copy_(w)         # restore
    agent.critic.bias.copy_(b)

print(f"p(correct) intact   : {acc_intact['p_correct']:.3f}")
print(f"p(correct) lesioned : {acc_lesioned['p_correct']:.3f}")

## Summary

| concept | NeuralRNN API | use when |
|---|---|---|
| environment | `make_env` / `register_env`, `"neurogym:<Task>"` passthrough | any gym-style task; neurogym tasks need no code |
| vectorized rollouts | `SyncVectorEnv` | always (`RLTrainer` expects it) |
| agent | `ActorCriticModel(ActorCriticConfig(core_config=..., action_dim=..., action_type=...))` | any RNN core; `"discrete"` / `"continuous"` / `"none"` (critic-only) |
| RL algorithm | `build_rl_loss("reinforce" or "ppo")`, `@register_rl_loss` | REINFORCE: simple, on-policy; PPO: sample-efficient |
| training | `RLTrainer(agent, envs, RLTrainingArguments(...), loss=..., eval_fn=...)` | on-policy training with BPTT |
| episode collection | `collect_episodes(agent, envs, n)` | behavior + hidden states for analysis |
| critic-only value learning | `TDObjective` + supervised `Trainer` | value-RNN paradigms on offline datasets |

1. An RL agent in NeuralRNN is just an RNN core with two extra heads — the whole analysis toolbox (fixed points, PCA, vector fields, ...) applies unchanged.
2. The env/loss/trainer layers are registry-based: new tasks and new algorithms plug in without touching the framework.
3. Reward-only training reproduces the behavioral *and* neural signatures of decision making; the critic is a teaching signal, not part of the decision machinery.

**Further reading** — full reproductions built on this layer:
`18_rl_rnn_paradigmA.ipynb` (economic choice, PPO + E-I core),
`19_value_rnn_paradigmA.ipynb` (value RNNs, critic-only TD),
`s1_plumetracknets.ipynb` (plume tracking, continuous PPO).